In [68]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import joblib
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMClassifier


In [69]:
df = pd.read_csv("../data/predictive_maintenance.csv")

print(df.head())
print(df.info())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

In [70]:
print(df.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


In [71]:
df = df.loc[:, ~df.columns.str.startswith('Product ID_')]

In [72]:
# Drop leakage columns
df.drop(columns=['TWF', 'HDF', 'PWF', 'OSF', 'RNF'], inplace=True, errors='ignore')

In [73]:
print(df.columns)

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure'],
      dtype='object')


In [74]:
X = df.drop('Machine failure', axis=1)
y = df['Machine failure']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [78]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ValueError: could not convert string to float: 'M18918'

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=5,  # imbalance handling
    eval_metric='logloss'
)

model.fit(X_train, y_train)

print("✅ Model Trained")

✅ Model Trained


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.984
ROC-AUC: 0.9734959201071733

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.77      0.75      0.76        68

    accuracy                           0.98      2000
   macro avg       0.88      0.87      0.88      2000
weighted avg       0.98      0.98      0.98      2000


Confusion Matrix:
 [[1917   15]
 [  17   51]]


In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=10,  # increase this
    eval_metric='logloss'
)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.984
ROC-AUC: 0.9734959201071733

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.77      0.75      0.76        68

    accuracy                           0.98      2000
   macro avg       0.88      0.87      0.88      2000
weighted avg       0.98      0.98      0.98      2000


Confusion Matrix:
 [[1917   15]
 [  17   51]]


In [ ]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)

In [ ]:
model.fit(X_train, y_train)
print(confusion_matrix(y_test, y_pred))

[[1905   27]
 [  15   53]]


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 0.9742570941420047

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.66      0.78      0.72        68

    accuracy                           0.98      2000
   macro avg       0.83      0.88      0.85      2000
weighted avg       0.98      0.98      0.98      2000


Confusion Matrix:
 [[1905   27]
 [  15   53]]


In [ ]:
# Temperature difference
df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]

# Power (approx mechanical stress)
df["power"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]

# Wear rate
df["wear_rate"] = df["Tool wear [min]"] / (df["Rotational speed [rpm]"] + 1)

C:\Users\Habibullah\AppData\Local\Temp\ipykernel_17820\1228102919.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]
C:\Users\Habibullah\AppData\Local\Temp\ipykernel_17820\1228102919.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["power"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]
C:\Users\Habibullah\AppData\Local\Temp\ipykernel_17820\1228102919.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of

In [ ]:
# Calculate imbalance ratio
neg, pos = np.bincount(y)
scale_weight = neg / pos

print("Scale Pos Weight:", scale_weight)

Scale Pos Weight: 28.49852507374631


In [ ]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    eval_metric='logloss'
)

In [ ]:

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 400],
    'scale_pos_weight': [scale_weight]
}

grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss'),
    param_grid,
    scoring='recall',  # 🔥 optimize for recall
    cv=3,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Params: {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 200, 'scale_pos_weight': np.float64(28.49852507374631)}


In [ ]:

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 400],
    'scale_pos_weight': [scale_weight]
}

grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss'),
    param_grid,
    scoring='recall',  # 🔥 optimize for recall
    cv=3,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Params: {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 200, 'scale_pos_weight': np.float64(28.49852507374631)}


In [ ]:
model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    class_weight='balanced'
)

In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 0.9691800633296797

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.90      0.95      1932
           1       0.25      0.96      0.40        68

    accuracy                           0.90      2000
   macro avg       0.62      0.93      0.67      2000
weighted avg       0.97      0.90      0.93      2000


Confusion Matrix:
 [[1736  196]
 [   3   65]]


In [ ]:
sample = [[295, 305, 1400, 35, 5, 1, 0]]  # safe machine

sample_scaled = scaler.transform(sample)

prob = best_model.predict_proba(sample_scaled)[0][1]
pred = 1 if prob > 0.3 else 0

print("Prediction:", pred)
print("Failure Probability:", prob)

Prediction: 0
Failure Probability: 0.12708426


c:\Users\Habibullah\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
sample = [[295, 305, 1400, 35, 5, 1, 0]]  # safe machine

sample_scaled = scaler.transform(sample)

prob = best_model.predict_proba(sample_scaled)[0][1]
pred = 1 if prob > 0.3 else 0

print("Prediction:", pred)
print("Failure Probability:", prob)

Prediction: 0
Failure Probability: 0.12708426


c:\Users\Habibullah\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
y_pred = (y_prob > 0.4).astype(int)

In [ ]:
threshold = 0.3  # or 0.4 if you choose balance

# Test set prediction
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("==== FINAL MODEL TEST ====")
print("Threshold:", threshold)
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

==== FINAL MODEL TEST ====
Threshold: 0.3
ROC-AUC: 0.9691800633296797

Confusion Matrix:
 [[1736  196]
 [   3   65]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.90      0.95      1932
           1       0.25      0.96      0.40        68

    accuracy                           0.90      2000
   macro avg       0.62      0.93      0.67      2000
weighted avg       0.97      0.90      0.93      2000



In [ ]:
test_cases = [
    [295, 305, 1400, 35, 5, 1, 0],     # Safe
    [300, 315, 1600, 50, 100, 0, 1],   # Medium
    [310, 330, 1800, 70, 250, 0, 1]    # High Risk
]

for sample in test_cases:
    sample_scaled = scaler.transform([sample])
    prob = best_model.predict_proba(sample_scaled)[0][1]
    pred = 1 if prob > threshold else 0

    print("Input:", sample)
    print("Prediction:", pred, "| Probability:", round(prob, 4))
    print("------")

Input: [295, 305, 1400, 35, 5, 1, 0]
Prediction: 0 | Probability: 0.1271
------
Input: [300, 315, 1600, 50, 100, 0, 1]
Prediction: 0 | Probability: 0.0909
------
Input: [310, 330, 1800, 70, 250, 0, 1]
Prediction: 1 | Probability: 0.8669
------


c:\Users\Habibullah\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Habibullah\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Habibullah\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
os.makedirs("app/models", exist_ok=True)

# Save paths
model_path = os.path.join("app/models", "model.pkl")
scaler_path = os.path.join("app/models", "scaler.pkl")

# Save files
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

print("✅ Model saved in app/models/")

✅ Model saved in root/models/
